# 🐦 Particle Swarm Optimization (PSO)
## Sürü Zekasına Dayalı Global Optimizasyon

## Teori

PSO, 1995 yılında Kennedy ve Eberhart tarafından geliştirilmiştir. **Kuş sürüleri** ve **balık okulları**nın kolektif davranışından ilham alır.

### Biyolojik İlham
Kuşlar yiyecek arayışında:
1. **Kendi deneyimini** hatırlar (nerede yiyecek buldu?)
2. **Sürü bilgisine** dikkat eder (sürünün en iyi konumu nerede?)
3. Mevcut **hızı/momentumu** ile hareket eder

### Matematiksel Model
Her parçacık için:

**Hız güncelleme:**
$$v_i^{t+1} = w \cdot v_i^t + c_1 r_1 (p_i - x_i^t) + c_2 r_2 (g - x_i^t)$$

**Konum güncelleme:**
$$x_i^{t+1} = x_i^t + v_i^{t+1}$$

| Sembol | Anlam | Rol |
|--------|-------|-----|
| $w$ | Atalet ağırlığı | Önceki hızı ne kadar koru? |
| $c_1$ | Bilişsel katsayı | Kendi en iyine ne kadar çekil? |
| $c_2$ | Sosyal katsayı | Sürünün en iyine ne kadar çekil? |
| $p_i$ | Kişisel en iyi (pbest) | Bu parçacığın tarihsel en iyisi |
| $g$ | Global en iyi (gbest) | Tüm sürünün en iyisi |
| $r_1, r_2$ | Rastgele sayılar | $U(0,1)$ — keşfi sağlar |

In [ ]:
# ============================================================
# KÜTÜPHANE İMPORTLARI
# ============================================================
import numpy as np                         # Sayısal işlemler
import matplotlib.pyplot as plt            # Statik grafikler
import matplotlib.animation as animation   # Animasyon
from matplotlib.colors import LogNorm      # Log renk ölçeği
from IPython.display import HTML, display
import warnings
warnings.filterwarnings('ignore')

# Tekrar üretilebilirlik
np.random.seed(42)
print("✅ Kütüphaneler yüklendi.")

## 1. Test Fonksiyonları

In [ ]:
# ============================================================
# TEST FONKSİYONLARI — PSO için Benchmark Seti
# ============================================================

def sphere(x):
    """Sphere: f(x) = Σ xi²  —  Minimum: 0 @ origin"""
    return np.sum(x**2)


def rastrigin(x):
    """
    Rastrigin — Çok-modlu, çok sayıda yerel minimum.
    PSO için zorlu: Yerel minimumlara takılabilir.
    Minimum: 0 @ (0,...,0)
    """
    A = 10
    return A * len(x) + np.sum(x**2 - A * np.cos(2 * np.pi * x))


def ackley(x):
    """
    Ackley — PSO'nun güçlü olduğu klasik test.
    Derin küresel minimum + çok sayıda yerel minimum.
    Minimum: 0 @ (0,...,0)
    """
    n = len(x)
    # İki bileşen: cos terimi (lokal yapı) + exp terimi (global yapı)
    sum_sq  = np.sum(x**2)
    sum_cos = np.sum(np.cos(2 * np.pi * x))
    return (
        -20 * np.exp(-0.2 * np.sqrt(sum_sq / n)) -
        np.exp(sum_cos / n) +
        20 + np.e
    )


def schwefel(x):
    """
    Schwefel — Yanıltıcı fonksiyon: Global minimum, diğer minimumlardan uzak!
    PSO sürü bilgisinin gücünü test eder.
    Minimum: 0 @ (420.97,...) — arama alanı [-500, 500]
    """
    n = len(x)
    return 418.9829 * n - np.sum(x * np.sin(np.sqrt(np.abs(x))))


# Fonksiyonları 2D görselleştir
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('PSO Test Fonksiyonları (2D)', fontsize=14, fontweight='bold')

functions = [
    (sphere,    'Sphere',    -5, 5,      'Blues'),
    (rastrigin, 'Rastrigin', -5.12, 5.12, 'viridis'),
    (ackley,    'Ackley',    -5, 5,      'plasma'),
    (schwefel,  'Schwefel',  -500, 500,  'hot'),
]

for ax, (fn, name, lb, ub, cmap) in zip(axes.flat, functions):
    x_range = np.linspace(lb, ub, 200)
    y_range = np.linspace(lb, ub, 200)
    X, Y = np.meshgrid(x_range, y_range)
    Z = np.array([[fn(np.array([X[i,j], Y[i,j]])) 
                   for j in range(200)] for i in range(200)])
    
    im = ax.contourf(X, Y, Z, levels=50, cmap=cmap)
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('pso_test_fonk.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Test fonksiyonları görselleştirildi.")

## 2. PSO İmplementasyonu

In [ ]:
# ============================================================
# PSO — TAM İMPLEMENTASYON
# ============================================================

class ParticleSwarm:
    """
    Particle Swarm Optimization (PSO) implementasyonu.
    
    Özellikler:
    - Lineer azalan atalet ağırlığı (w)
    - Hız sınırlama
    - Sınır kontrolü (reflection/clamping)
    """
    
    def __init__(
        self,
        n_particles = 30,      # Sürüdeki parçacık sayısı
        n_dims      = 2,       # Problem boyutu
        lower_bound = -5.12,   # Arama alanı alt sınırı
        upper_bound =  5.12,   # Arama alanı üst sınırı
        w_max       = 0.9,     # Maksimum atalet ağırlığı (başlangıçta)
        w_min       = 0.4,     # Minimum atalet ağırlığı (sonda)
        c1          = 2.05,    # Bilişsel (personal) katsayı
        c2          = 2.05,    # Sosyal (social) katsayı
    ):
        self.n_p = n_particles
        self.n   = n_dims
        self.lb  = lower_bound
        self.ub  = upper_bound
        self.w_max = w_max; self.w_min = w_min
        self.c1 = c1; self.c2 = c2
        
        # Maksimum hız: Arama alanının %20'si
        # Neden? Çok büyük hız → arama alanı dışına çıkar
        # Çok küçük hız → yavaş keşif
        self.v_max = 0.2 * (upper_bound - lower_bound)
        
        # ── PARÇACIKLARI BAŞLAT ──────────────────────────────
        # Konum: Arama alanında uniform dağılım
        self.positions = np.random.uniform(
            self.lb, self.ub, (n_particles, n_dims)
        )
        
        # Hız: Başlangıçta küçük rastgele hızlar
        # Neden küçük? İlk adımda parçacıkların sınır dışına çıkmaması için
        v_range = self.v_max * 0.1
        self.velocities = np.random.uniform(
            -v_range, v_range, (n_particles, n_dims)
        )
        
        # ── KİŞİSEL EN İYİLER (PBEST) ───────────────────────
        # Her parçacığın kendi tarihsel en iyi konumu
        self.pbest_pos = self.positions.copy()    # Başlangıçta mevcut konum
        self.pbest_val = np.full(n_particles, np.inf)  # Henüz değerlendirilmedi
        
        # ── GLOBAL EN İYİ (GBEST) ────────────────────────────
        # Tüm sürünün şimdiye kadar bulduğu en iyi konum
        self.gbest_pos = np.zeros(n_dims)
        self.gbest_val = np.inf
        
        print(f"PSO başlatıldı: {n_particles} parçacık, {n_dims}D")
        print(f"  Hız sınırı: ±{self.v_max:.3f}")
        print(f"  Atalet: {w_max} → {w_min} (lineer azalan)")
    
    
    def evaluate(self, fitness_fn):
        """
        Tüm parçacıkları değerlendir ve pbest/gbest güncelle.
        
        Parametreler:
            fitness_fn (callable): Minimize edilecek fonksiyon
        
        Döndürür:
            np.ndarray: Her parçacığın fitness değeri
        """
        fitness = np.array([fitness_fn(p) for p in self.positions])
        
        # ── PBEST GÜNCELLEME ─────────────────────────────────
        # Eğer mevcut konum pbest'ten iyiyse güncelle
        improved = fitness < self.pbest_val    # Boolean mask
        self.pbest_val = np.where(improved, fitness, self.pbest_val)
        self.pbest_pos[improved] = self.positions[improved].copy()
        
        # ── GBEST GÜNCELLEME ─────────────────────────────────
        # Tüm parçacıklar arasında en iyi olanı bul
        best_idx = np.argmin(self.pbest_val)
        if self.pbest_val[best_idx] < self.gbest_val:
            self.gbest_val = self.pbest_val[best_idx]
            self.gbest_pos = self.pbest_pos[best_idx].copy()
        
        return fitness
    
    
    def update(self, current_gen, max_gens):
        """
        Hız ve konum güncelleme — PSO'nun kalbi.
        
        Parametreler:
            current_gen (int): Mevcut nesil (atalet azaltması için)
            max_gens    (int): Toplam nesil sayısı
        """
        # ── ATALET AĞIRLIĞI — Lineer Azaltma ─────────────────
        # Başlangıçta büyük w → geniş keşif (exploration)
        # Sonunda küçük w   → hassas sömürü (exploitation)
        # Bu trade-off PSO'nun temel dengesidir!
        w = self.w_max - (self.w_max - self.w_min) * current_gen / max_gens
        
        # ── RASTGELE FAKTÖRLER ───────────────────────────────
        # r1, r2: [0,1] aralığında uniform rastgele
        # Her parçacık ve boyut için ayrı r1, r2 → çeşitlilik sağlar
        r1 = np.random.rand(self.n_p, self.n)    # Bilişsel için
        r2 = np.random.rand(self.n_p, self.n)    # Sosyal için
        
        # ── HIZ GÜNCELLEMESİ ─────────────────────────────────
        # v_new = w·v_old + c1·r1·(pbest - x) + c2·r2·(gbest - x)
        #
        # Bileşen 1: w·v_old          → Atalet (momentum)
        # Bileşen 2: c1·r1·(pbest-x)  → Kişisel çekim (hafıza)
        # Bileşen 3: c2·r2·(gbest-x)  → Sosyal çekim (sürü bilgisi)
        
        inertia   = w * self.velocities                         # Bileşen 1
        cognitive = self.c1 * r1 * (self.pbest_pos - self.positions)  # Bileşen 2
        social    = self.c2 * r2 * (self.gbest_pos - self.positions)  # Bileşen 3
        
        self.velocities = inertia + cognitive + social
        
        # ── HIZ SINIRLAMASI ──────────────────────────────────
        # Neden? Çok büyük hız → parçacıklar sınır dışına savrulur
        # Çözüm: Hızı [-v_max, v_max] aralığına kırp
        self.velocities = np.clip(self.velocities, -self.v_max, self.v_max)
        
        # ── KONUM GÜNCELLEMESİ ──────────────────────────────
        # x_new = x_old + v_new
        self.positions = self.positions + self.velocities
        
        # ── SINIR KONTROLÜ (Reflection) ──────────────────────
        # Sınır dışına çıkan parçacıkları yansıt ve hızı ters çevir
        # Alternatif: Kırpma (basit ama hız yönünü bozar)
        out_lower = self.positions < self.lb    # Alt sınır ihlali
        out_upper = self.positions > self.ub    # Üst sınır ihlali
        
        # Yansıtma: Sınır içine geri getir + hızı ters çevir
        self.positions = np.where(out_lower, self.lb, self.positions)
        self.positions = np.where(out_upper, self.ub, self.positions)
        self.velocities = np.where(out_lower | out_upper, 
                                    -self.velocities, self.velocities)


print("✅ PSO sınıfı tanımlandı.")

## 3. Ana PSO Döngüsü

In [ ]:
# ============================================================
# ANA PSO OPTİMİZASYON DÖNGÜSÜ
# ============================================================

def run_pso(
    fitness_fn,
    n_dims      = 2,
    n_particles = 30,
    max_iters   = 300,
    lower       = -5.12,
    upper       =  5.12,
    w_max       = 0.9,
    w_min       = 0.4,
    c1          = 2.05,
    c2          = 2.05,
    track_positions = False,    # Parçacık konumlarını kaydet (animasyon için)
    verbose     = True
):
    """
    PSO optimizasyon döngüsünü çalıştırır.
    
    Döndürür:
        gbest_pos (np.ndarray): En iyi çözüm
        gbest_val (float)     : En iyi fitness
        history   (dict)      : İlerleme kaydı
    """
    # PSO sürüsünü başlat
    swarm = ParticleSwarm(
        n_particles=n_particles, n_dims=n_dims,
        lower_bound=lower, upper_bound=upper,
        w_max=w_max, w_min=w_min, c1=c1, c2=c2
    )
    
    history = {
        'gbest'     : [],    # Global en iyi fitness
        'mean_fit'  : [],    # Ortalama fitness
        'diversity' : [],    # Sürü çeşitliliği (std)
        'positions' : [],    # Parçacık konumları (opsiyonel)
        'gbest_pos' : [],    # Global en iyi konumu
    }
    
    print(f"\nPSO başlatılıyor...")
    
    for it in range(max_iters):
        
        # ── 1. DEĞERLENDİRME ────────────────────────────────
        # Tüm parçacıkları değerlendir, pbest/gbest güncelle
        fitness = swarm.evaluate(fitness_fn)
        
        # ── 2. İSTATİSTİK KAYIT ─────────────────────────────
        # Sürü çeşitliliği: Parçacıkların gbest'e olan ortalama uzaklığı
        dists_to_gbest = np.linalg.norm(
            swarm.positions - swarm.gbest_pos, axis=1
        )
        
        history['gbest'].append(swarm.gbest_val)
        history['mean_fit'].append(np.mean(fitness))
        history['diversity'].append(np.mean(dists_to_gbest))
        history['gbest_pos'].append(swarm.gbest_pos.copy())
        
        if track_positions and n_dims == 2:
            # 2D durumda parçacık konumlarını kaydet
            history['positions'].append(swarm.positions.copy())
        
        # ── 3. GÜNCELLEME ────────────────────────────────────
        # Hız ve konum güncelle
        swarm.update(current_gen=it, max_gens=max_iters)
        
        # Loglama
        if verbose and (it < 5 or (it+1) % 50 == 0):
            w_current = swarm.w_max - (swarm.w_max - swarm.w_min) * it / max_iters
            print(f"İter {it+1:4d} | gbest={swarm.gbest_val:.6f} | "
                  f"mean={np.mean(fitness):.4f} | diversity={np.mean(dists_to_gbest):.3f} | "
                  f"w={w_current:.3f}")
    
    return swarm.gbest_pos, swarm.gbest_val, history


# ── ÇALIŞTIR ─────────────────────────────────────────────────
print("🐦 PSO — Rastrigin Optimizasyonu")
best_pos, best_val, hist = run_pso(
    fitness_fn = rastrigin,
    n_dims      = 2,
    n_particles = 40,
    max_iters   = 200,
    lower       = -5.12,
    upper       =  5.12,
    track_positions = True
)

print(f"\n{'='*55}")
print(f"✅ Optimizasyon tamamlandı!")
print(f"   En iyi konum : {best_pos}")
print(f"   Fitness      : {best_val:.8f}")
print(f"   Teorik min   : 0.0 @ (0, 0)")

In [ ]:
# ============================================================
# SONUÇ GÖRSELLEŞTİRME
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('PSO — Optimizasyon Analizi (Rastrigin)', fontsize=14, fontweight='bold')

iters = np.arange(1, len(hist['gbest']) + 1)

# ── Graf 1: Konverjans ────────────────────────────────────
axes[0,0].semilogy(iters, hist['gbest'], 'b-', linewidth=2, label='Global en iyi')
axes[0,0].semilogy(iters, hist['mean_fit'], 'r--', linewidth=1.5, alpha=0.7, label='Ortalama')
axes[0,0].set_xlabel('İterasyon'); axes[0,0].set_ylabel('Fitness [log]')
axes[0,0].set_title('Konverjans Eğrisi')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3, which='both')

# ── Graf 2: Sürü Çeşitliliği ─────────────────────────────
axes[0,1].plot(iters, hist['diversity'], 'g-', linewidth=2)
axes[0,1].set_xlabel('İterasyon'); axes[0,1].set_ylabel('Gbest\'e Ortalama Uzaklık')
axes[0,1].set_title('Sürü Çeşitliliği')
axes[0,1].grid(True, alpha=0.3)
# Çeşitlilik azalması → sürü gbest etrafında toplanıyor
axes[0,1].fill_between(iters, hist['diversity'], alpha=0.3, color='green')

# ── Graf 3: Parçacık İzi ──────────────────────────────────
if hist['positions']:
    # Rastrigin arka planı
    x_r = np.linspace(-5.12, 5.12, 200)
    y_r = np.linspace(-5.12, 5.12, 200)
    Xr, Yr = np.meshgrid(x_r, y_r)
    Zr = np.array([[rastrigin(np.array([Xr[i,j], Yr[i,j]])) 
                    for j in range(200)] for i in range(200)])
    
    axes[1,0].contourf(Xr, Yr, Zr, levels=40, cmap='viridis', alpha=0.6)
    
    # Farklı zaman adımlarında parçacık konumları
    snapshot_iters = [0, 20, 50, 100, 150, 199]
    colors_it = plt.cm.cool(np.linspace(0, 1, len(snapshot_iters)))
    
    for idx, (snap_i, col) in enumerate(zip(snapshot_iters, colors_it)):
        if snap_i < len(hist['positions']):
            pos_snap = hist['positions'][snap_i]
            axes[1,0].scatter(
                pos_snap[:, 0], pos_snap[:, 1],
                color=col, s=20, alpha=0.8,
                label=f'İter {snap_i+1}'
            )
    
    axes[1,0].scatter([0], [0], color='red', s=200, marker='*', zorder=10, label='Minimum')
    axes[1,0].set_xlabel('x₁'); axes[1,0].set_ylabel('x₂')
    axes[1,0].set_title('Parçacık Konumları (Farklı İterasyonlar)')
    axes[1,0].legend(fontsize=7, ncol=2)

# ── Graf 4: Gbest İzi ─────────────────────────────────────
gbest_positions = np.array(hist['gbest_pos'])
axes[1,1].contourf(Xr, Yr, Zr, levels=40, cmap='viridis', alpha=0.6)

# Gbest'in zaman içindeki yolculuğu
sc = axes[1,1].scatter(
    gbest_positions[:, 0], gbest_positions[:, 1],
    c=iters, cmap='hot', s=10, alpha=0.8
)
plt.colorbar(sc, ax=axes[1,1], label='İterasyon')
axes[1,1].scatter([0], [0], color='cyan', s=200, marker='*', zorder=10, label='Teorik min')
axes[1,1].set_xlabel('x₁'); axes[1,1].set_ylabel('x₂')
axes[1,1].set_title('Global En İyi Konumunun Evrimi')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('pso_sonuclar.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. PSO Varyantları ve Parametr Analizi

In [ ]:
# ============================================================
# PSO PARAMETRE ANALİZİ — c1 vs c2 Dengesi
# ============================================================
#
# c1 (bilişsel): Kendi deneyimine ne kadar güven?
# c2 (sosyal)  : Sürü bilgisine ne kadar güven?
#
# Uç durumlar:
# c1 >> c2 → Her parçacık kendi pbest'ine koşar → çeşitlilik yüksek, yavaş
# c2 >> c1 → Tüm sürü hızla gbest'e toplanır → erken yakınsama riski
# c1 = c2  → Dengeli keşif ve sömürü

param_configs = [
    {'c1': 2.5, 'c2': 0.5,  'label': 'Bilişsel ağırlıklı (c1>>c2)'},
    {'c1': 2.0, 'c2': 2.0,  'label': 'Dengeli (c1=c2)'},
    {'c1': 0.5, 'c2': 2.5,  'label': 'Sosyal ağırlıklı (c2>>c1)'},
    {'c1': 1.5, 'c2': 1.5,  'label': 'Küçük katsayılar'},
]

n_runs = 5    # İstatistiksel güvenilirlik için
results_params = {}

print(f"{'Yapılandırma':40s} | {'Ort. Best':>10} | {'Std':>10}")
print("-" * 68)

for cfg in param_configs:
    run_results = []
    for run in range(n_runs):
        np.random.seed(run * 7)
        _, bv, _ = run_pso(
            fitness_fn=rastrigin, n_dims=2,
            n_particles=30, max_iters=150,
            c1=cfg['c1'], c2=cfg['c2'],
            verbose=False
        )
        run_results.append(bv)
    
    results_params[cfg['label']] = run_results
    print(f"{cfg['label']:40s} | {np.mean(run_results):>10.4f} | {np.std(run_results):>10.4f}")


# Görselleştirme
fig, ax = plt.subplots(figsize=(10, 5))

labels  = list(results_params.keys())
data    = list(results_params.values())
bp = ax.boxplot(data, labels=[l.split('(')[0].strip() for l in labels], patch_artist=True)

colors = ['#ff7f7f', '#7fbf7f', '#7f7fff', '#ffbf7f']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_ylabel('En İyi Fitness (5 çalışma)')
ax.set_title('c1/c2 Parametrelerinin PSO Performansına Etkisi\n(Rastrigin 2D)')
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Teorik optimum')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pso_parametre_analiz.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# BOYUT ÖLÇEKLENEBİLİRLİĞİ VE KARŞILAŞTIRMA
# ============================================================

dims_test = [2, 5, 10, 20]
funcs_test = [
    (rastrigin, 'Rastrigin', -5.12, 5.12),
    (ackley,    'Ackley',    -5.0,  5.0),
]

print("Boyut ölçeklenebilirliği analizi...\n")
print(f"{'Fonksiyon':12s} | {'Boyut':>5} | {'En İyi':>12} | {'Hata':>12}")
print("-" * 50)

scale_results = {}

for fn, fn_name, lb, ub in funcs_test:
    scale_results[fn_name] = {'dims': [], 'bests': []}
    
    for d in dims_test:
        np.random.seed(42)
        _, bv, _ = run_pso(
            fitness_fn=fn, n_dims=d,
            n_particles=max(20, 5*d),    # Boyuta göre parçacık sayısı
            max_iters=300,
            lower=lb, upper=ub,
            verbose=False
        )
        scale_results[fn_name]['dims'].append(d)
        scale_results[fn_name]['bests'].append(bv)
        print(f"{fn_name:12s} | {d:>5} | {bv:>12.4e} | {bv:>12.4e}")


# Görselleştirme
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('PSO Boyut Ölçeklenebilirliği', fontsize=13, fontweight='bold')

markers = ['o-', 's-']
colors_fn = ['#2196F3', '#F44336']

for ax in axes:
    for (fn_name, res), mk, clr in zip(scale_results.items(), markers, colors_fn):
        ax.semilogy(res['dims'], res['bests'], mk, color=clr,
                   linewidth=2, markersize=8, label=fn_name)
    ax.set_xlabel('Problem Boyutu (n)')
    ax.set_ylabel('En İyi Fitness [log]')
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[0].set_title('Tüm Boyutlar')
axes[1].set_title('Yüksek Boyut Odaklı')
axes[1].set_xlim(5, 21)

plt.tight_layout()
plt.savefig('pso_boyut.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📌 Not: PSO, boyut arttıkça daha fazla parçacık ve iterasyon gerektirir.")

## 5. Üç Algoritmanın Karşılaştırması

In [ ]:
# ============================================================
# GA vs CMA-ES vs PSO — KARŞILAŞTIRMA
# ============================================================

print("="*60)
print("3 Algoritma Karşılaştırması — Rastrigin 10D")
print("="*60)

N_RUNS = 5
N_DIMS = 10

# ── Sonuçlar ─────────────────────────────────────────────────
comparison = {'PSO': [], 'GA (Basit)': []}

for run in range(N_RUNS):
    np.random.seed(run)
    
    # PSO
    _, pso_best, _ = run_pso(
        rastrigin, n_dims=N_DIMS, n_particles=50,
        max_iters=500, lower=-5.12, upper=5.12, verbose=False
    )
    comparison['PSO'].append(pso_best)

# Basit GA implementasyonu (karşılaştırma için)
def simple_ga_run(n_dims=10, n_runs=5):
    results = []
    for run in range(n_runs):
        np.random.seed(run)
        pop = np.random.uniform(-5.12, 5.12, (100, n_dims))
        for _ in range(500):
            fit = np.array([rastrigin(x) for x in pop])
            # Elit seçim
            idx = np.argsort(fit)[:50]
            parents = pop[idx]
            # Çaprazlama
            new_pop = []
            for i in range(0, len(parents)-1, 2):
                pt = np.random.randint(1, n_dims)
                c1b = np.concatenate([parents[i][:pt], parents[i+1][pt:]])
                c2b = np.concatenate([parents[i+1][:pt], parents[i][pt:]])
                new_pop.extend([c1b, c2b])
            new_pop = np.array(new_pop[:50])
            # Mutasyon
            mask = np.random.rand(*new_pop.shape) < 0.1
            new_pop += mask * np.random.randn(*new_pop.shape) * 0.5
            new_pop = np.clip(new_pop, -5.12, 5.12)
            pop = np.vstack([parents[:10], new_pop])[:100]
        fit = np.array([rastrigin(x) for x in pop])
        results.append(np.min(fit))
    return results

comparison['GA (Basit)'] = simple_ga_run(N_DIMS, N_RUNS)

# Sonuçları göster
print(f"\n{'Algoritma':20s} | {'Ort.':>12} | {'Std':>10} | {'En İyi':>10}")
print("-" * 60)
for alg, vals in comparison.items():
    print(f"{alg:20s} | {np.mean(vals):>12.4f} | {np.std(vals):>10.4f} | {np.min(vals):>10.4f}")

# Box plot
fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot(
    list(comparison.values()),
    labels=list(comparison.keys()),
    patch_artist=True
)
box_colors = ['#66B2FF', '#FF9999']
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)

ax.set_ylabel('En İyi Fitness (5 çalışma)')
ax.set_title(f'PSO vs GA Karşılaştırması\n(Rastrigin {N_DIMS}D, 500 iterasyon)')
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Teorik optimum')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pso_vs_ga.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Özet ve Değerlendirme

### PSO'nun Güçlü Yönleri ✅
- **Basit implementasyon**: GA'ya göre daha az operatör
- **Hızlı yakınsama**: İyi parametrelerle hızlı global arama
- **Az hiperparametre**: Genellikle w, c1, c2 yeterli
- **Sürekli uzayda etkili**: Gradient-free optimizasyon

### Zayıf Yönler ⚠️
- **Erken yakınsama**: Sürü çok hızlı gbest'e toplanabilir
- **Ayrıksal problemlere uygun değil**: Binary PSO varyantı gerektirir
- **Büyük boyutlarda zayıf**: 100+ boyut için CMA-ES daha iyi

### GA vs CMA-ES vs PSO Seçim Rehberi

| Kriter | GA | CMA-ES | PSO |
|--------|----|---------|---------|
| Boyut | Orta (2-50) | Yüksek (10-1000) | Düşük-Orta (2-50) |
| Ayrıksal | ✅ | ❌ | Kısıtlı |
| Çok-modlu | ✅ | Kısıtlı | ✅ |
| Hız | Yavaş | Hızlı | Orta |
| Implementasyon | Orta | Zor | Kolay |